# CPE-Identifier: Full Pipeline Notebook

End-to-end pipeline for automated CPE extraction from CVE descriptions using transformer-based NER.

**Pipeline steps:**
1. Environment setup & GPU check
2. Google Drive mount (optional persistence)
3. Repository clone & dependency install
4. Data download from NVD API
5. Data exploration & label statistics
6. Model training (BERT / XLNet / GPT-2)
7. TensorBoard monitoring
8. Test set evaluation & per-entity metrics
9. Inference on custom CVE text
10. Batch inference & export
11. Multi-model comparison
12. Save artifacts to Google Drive

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---
## 0. Environment Setup

In [ ]:
import subprocess, sys, os

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No NVIDIA GPU detected. Training will run on CPU (slow).')
    print('Go to Runtime → Change runtime type → T4 GPU')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 1. Google Drive Mount

Mount Drive to persist data, checkpoints and results across sessions.
Skip this cell if you don't need persistence (data will be lost when runtime resets).

In [ ]:
USE_DRIVE = True   # Set False to skip Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/cpe_identifier'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'Drive mounted. Artifacts will be saved to: {DRIVE_ROOT}')
else:
    DRIVE_ROOT = None
    print('Drive not mounted. Artifacts stored in /content only.')

---
## 2. Clone Repository & Install Dependencies

In [ ]:
REPO_URL  = 'https://github.com/lowgame/cpe_indentifier.git'
REPO_DIR  = '/content/cpe_indentifier'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repository already present — pulled latest.')

%cd {REPO_DIR}

In [ ]:
%%capture install_log
!pip install -r requirements.txt
# Download NLTK tokenizer data (used by CVEPreprocessor)
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Installation complete.')

In [ ]:
# Verify key packages
import importlib
pkgs = ['torch', 'transformers', 'seqeval', 'nltk', 'tqdm', 'tensorboard']
for pkg in pkgs:
    ver = importlib.import_module(pkg).__version__
    print(f'  {pkg:<15} {ver}')

---
## 3. Configuration

Set your NVD API key and hyperparameters here.

In [ ]:
# ─── NVD API ──────────────────────────────────────────────────────────────────
# Get a free API key at https://nvd.nist.gov/developers/request-an-api-key
# With key: ~50 req/30s (10x faster). Without: ~5 req/30s.
NVD_API_KEY = ''   # Leave empty to use unauthenticated (slower)

# ─── Data range ───────────────────────────────────────────────────────────────
START_YEAR = 2020
END_YEAR   = 2022   # Increase for more data (2020-2023 ≈ 90k CVEs)

# ─── Training ─────────────────────────────────────────────────────────────────
MODEL      = 'bert'   # 'bert' | 'xlnet' | 'gpt2'
EPOCHS     = 5        # Paper uses 20; use 3-5 for a quick run on Colab
BATCH_SIZE = 32       # T4 (16 GB) handles 32 comfortably
LR         = 2e-5
NUM_WORKERS= 2
SEED       = 42

# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR       = f'{REPO_DIR}/data'
ANNOTATED_FILE = f'{DATA_DIR}/annotated/cves_{START_YEAR}_{END_YEAR}.bio'
MODELS_DIR     = f'{REPO_DIR}/models'
LOGS_DIR       = f'{REPO_DIR}/logs'
CHECKPOINT_DIR = f'{MODELS_DIR}/{MODEL}_ner/best'

# Expose API key to subprocesses
if NVD_API_KEY:
    os.environ['NVD_API_KEY'] = NVD_API_KEY

print('Configuration:')
print(f'  Model      : {MODEL}')
print(f'  Years      : {START_YEAR}–{END_YEAR}')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  API key    : {"set" if NVD_API_KEY else "not set (slower)"}')

---
## 4. Download CVE Data from NVD API

Fetches CVEs from NVD API 2.0, auto-splits requests into ≤119-day chunks (API limit),
caches responses locally, then generates BIO-tagged training data.

In [ ]:
import os

# If Drive is mounted, symlink data dir so cache persists across sessions
if USE_DRIVE and DRIVE_ROOT:
    drive_data = os.path.join(DRIVE_ROOT, 'data')
    os.makedirs(drive_data, exist_ok=True)
    if not os.path.islink(f'{REPO_DIR}/data'):
        os.symlink(drive_data, f'{REPO_DIR}/data')
    print(f'Data directory linked to Drive: {drive_data}')

os.makedirs(f'{DATA_DIR}/raw/nvd_cache', exist_ok=True)
os.makedirs(f'{DATA_DIR}/annotated', exist_ok=True)
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)

In [ ]:
api_key_arg = f'--api-key {NVD_API_KEY}' if NVD_API_KEY else ''

!python scripts/download_data.py \
    --start-year {START_YEAR} \
    --end-year   {END_YEAR}   \
    {api_key_arg}

---
## 5. Explore the Dataset

In [ ]:
import json
import pandas as pd

raw_file = f'{DATA_DIR}/raw/cves_{START_YEAR}_{END_YEAR}.jsonl'
cves = [json.loads(l) for l in open(raw_file)]
print(f'Total CVEs      : {len(cves):,}')
print(f'With CPE matches: {sum(1 for c in cves if c.get("cpe_matches")):,}')

df = pd.DataFrame([{
    'id'       : c['id'],
    'severity' : c.get('severity', 'UNKNOWN'),
    'published': c.get('published', '')[:10],
    'desc_len' : len(c.get('description', '')),
    'n_cpes'   : len(c.get('cpe_matches', [])),
} for c in cves])

print('\nSeverity distribution:')
print(df['severity'].value_counts().to_string())

print('\nSample CVEs:')
df.head(5)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.data.annotator import BIOAnnotator

sequences = BIOAnnotator.load_bio_file(ANNOTATED_FILE)
print(f'Annotated sequences : {len(sequences):,}')

from src.data.annotator import BIOAnnotator
ann = BIOAnnotator()
stats = ann.get_label_statistics(sequences)

print('\nLabel distribution:')
for label, count in sorted(stats.items(), key=lambda x: -x[1]):
    bar = '█' * (count * 40 // max(stats.values()))
    print(f'  {label:<12} {count:>10,}  {bar}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Exclude 'O' for readability
entity_stats = {k: v for k, v in stats.items() if k != 'O'}
labels_sorted = sorted(entity_stats, key=entity_stats.get, reverse=True)
counts_sorted = [entity_stats[l] for l in labels_sorted]

colors = {
    'B-VENDOR': '#FF6B6B', 'I-VENDOR': '#FF9999',
    'B-PRODUCT': '#4ECDC4','I-PRODUCT': '#80E8E2',
    'B-VERSION': '#45B7D1','I-VERSION': '#80D4E8',
    'B-UPDATE':  '#96CEB4','I-UPDATE':  '#C0E4D0',
    'B-EDITION': '#FFEAA7','I-EDITION': '#FFF3CC',
}
bar_colors = [colors.get(l, '#AAAAAA') for l in labels_sorted]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_sorted, counts_sorted, color=bar_colors)
ax.set_xlabel('Count')
ax.set_title(f'Entity Label Distribution (CVEs {START_YEAR}–{END_YEAR})')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, cnt in zip(bars, counts_sorted):
    ax.text(bar.get_width() + max(counts_sorted)*0.005, bar.get_y() + bar.get_height()/2,
            f'{cnt:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Print a few annotated examples
print('Sample annotated sequences:\n')
for i, seq in enumerate(sequences[:3]):
    tokens  = [w for w, _ in seq]
    labels  = [l for _, l in seq]
    entity_tokens = [(w, l) for w, l in seq if l != 'O']
    print(f'--- Sequence {i+1} ---')
    print(' '.join(tokens[:20]) + ('...' if len(tokens) > 20 else ''))
    print('Entities:', entity_tokens)
    print()

---
## 6. Train the Model

Fine-tunes a transformer on the annotated BIO data.

| Model | F1 (paper) | Notes |
|-------|-----------|-------|
| **bert** | **95.48%** | Best overall, bidirectional |
| xlnet | 94.21% | Permutation LM, slightly slower |
| gpt2  | 91.43% | Causal attention, worst for NER |

In [ ]:
# Start TensorBoard before training so you can watch live
%load_ext tensorboard
%tensorboard --logdir {LOGS_DIR}

In [ ]:
!python scripts/train.py \
    --model        {MODEL}          \
    --data         {ANNOTATED_FILE} \
    --epochs       {EPOCHS}         \
    --batch-size   {BATCH_SIZE}     \
    --lr           {LR}             \
    --num-workers  {NUM_WORKERS}    \
    --seed         {SEED}           \
    --save-dir     {MODELS_DIR}     \
    --log-dir      {LOGS_DIR}

### Optional: train all three models for comparison

In [ ]:
TRAIN_ALL = False   # Set True to train bert + xlnet + gpt2

if TRAIN_ALL:
    for m in ['bert', 'xlnet', 'gpt2']:
        print(f'\n{"="*60}')
        print(f'Training {m.upper()}...')
        print('='*60)
        !python scripts/train.py \
            --model        {m}              \
            --data         {ANNOTATED_FILE} \
            --epochs       {EPOCHS}         \
            --batch-size   {BATCH_SIZE}     \
            --num-workers  {NUM_WORKERS}    \
            --seed         {SEED}           \
            --save-dir     {MODELS_DIR}     \
            --log-dir      {LOGS_DIR}

---
## 7. Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

results_path = f'{MODELS_DIR}/{MODEL}_ner/results.json'
with open(results_path) as f:
    results = json.load(f)

history = results['history']
epochs  = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'{MODEL.upper()}-NER Training History', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs, history['val_f1'], 'g-o', ms=4)
best_f1_epoch = history['val_f1'].index(max(history['val_f1'])) + 1
axes[1].axvline(best_f1_epoch, color='orange', linestyle='--', label=f'Best @ epoch {best_f1_epoch}')
axes[1].set_title('Validation F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Precision & Recall
axes[2].plot(epochs, history['val_precision'], 'm-o', ms=4, label='Precision')
axes[2].plot(epochs, history['val_recall'],    'c-o', ms=4, label='Recall')
axes[2].set_title('Precision & Recall (Val)')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nBest validation F1 : {max(history["val_f1"]):.4f} (epoch {best_f1_epoch})')

---
## 8. Test Set Evaluation

In [ ]:
test_metrics = results['test_metrics']

print(f'Test Set Results — {MODEL.upper()}-NER')
print('─' * 35)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<12} {v:.4f}')

In [ ]:
# Full evaluation using evaluate.py (produces per-entity breakdown)
!python scripts/evaluate.py \
    --model      {MODEL}          \
    --checkpoint {CHECKPOINT_DIR} \
    --data       {ANNOTATED_FILE} \
    --output     {MODELS_DIR}/{MODEL}_ner/eval_results.json

In [ ]:
eval_path = f'{MODELS_DIR}/{MODEL}_ner/eval_results.json'
if os.path.exists(eval_path):
    with open(eval_path) as f:
        eval_data = json.load(f)

    per_entity = eval_data.get('per_entity', {})
    if per_entity:
        rows = []
        for ent, m in per_entity.items():
            rows.append({'Entity': ent, 'Precision': m.get('precision', 0),
                         'Recall': m.get('recall', 0), 'F1': m.get('f1', 0)})
        ent_df = pd.DataFrame(rows).set_index('Entity').sort_values('F1', ascending=False)
        display(ent_df.style.format('{:.4f}').background_gradient(cmap='Greens'))

        # Bar chart
        fig, ax = plt.subplots(figsize=(9, 4))
        x = range(len(ent_df))
        w = 0.25
        ax.bar([i-w for i in x], ent_df['Precision'], width=w, label='Precision', color='steelblue')
        ax.bar([i   for i in x], ent_df['F1'],        width=w, label='F1',        color='seagreen')
        ax.bar([i+w for i in x], ent_df['Recall'],    width=w, label='Recall',    color='tomato')
        ax.set_xticks(list(x))
        ax.set_xticklabels(ent_df.index, rotation=30, ha='right')
        ax.set_ylim(0, 1.05)
        ax.set_title(f'Per-Entity Metrics — {MODEL.upper()}-NER')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

---
## 9. Inference on Custom CVE Text

In [ ]:
from src.inference.predictor import CPEPredictor

predictor = CPEPredictor.from_checkpoint(
    checkpoint_dir=CHECKPOINT_DIR,
    model_type=MODEL,
)
print(f'Predictor loaded from {CHECKPOINT_DIR}')

In [ ]:
# Change this to any CVE description
CVE_TEXT = """Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 
2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do 
not protect against attacker controlled LDAP and other JNDI related endpoints."""

result = predictor.predict(CVE_TEXT)

print('=' * 60)
print('PREDICTION RESULT')
print('=' * 60)
print(result.summary())
print(f'\nConfidence : {result.confidence:.2%}')

In [ ]:
# Colorized HTML output in Colab
from IPython.display import HTML, display

ENTITY_COLORS = {
    'VENDOR':  '#FF6B6B',
    'PRODUCT': '#4ECDC4',
    'VERSION': '#45B7D1',
    'UPDATE':  '#96CEB4',
    'EDITION': '#FFEAA7',
}

def render_bio(tokens, bio_labels):
    html_parts = ['<div style="font-family:monospace;font-size:14px;line-height:2">']
    i = 0
    while i < len(tokens):
        label = bio_labels[i] if i < len(bio_labels) else 'O'
        if label.startswith('B-'):
            etype = label[2:]
            color = ENTITY_COLORS.get(etype, '#EEE')
            span_tokens = [tokens[i]]
            i += 1
            while i < len(tokens) and (i < len(bio_labels)) and bio_labels[i] == f'I-{etype}':
                span_tokens.append(tokens[i])
                i += 1
            span_text = ' '.join(span_tokens)
            html_parts.append(
                f'<mark style="background:{color};padding:2px 4px;border-radius:3px;margin:1px" '
                f'title="{etype}"><b>{span_text}</b> <sup style="font-size:9px">{etype}</sup></mark> '
            )
        else:
            html_parts.append(f'{tokens[i]} ')
            i += 1
    html_parts.append('</div>')
    return ''.join(html_parts)

display(HTML(render_bio(result.tokens, result.bio_labels)))

# Legend
legend = ' '.join(
    f'<span style="background:{c};padding:2px 6px;border-radius:3px;margin:2px;font-size:12px">{e}</span>'
    for e, c in ENTITY_COLORS.items()
)
display(HTML(f'<div style="margin-top:8px">{legend}</div>'))

In [ ]:
# Famous CVE examples
FAMOUS_CVES = [
    ('CVE-2021-44228 (Log4Shell)',
     'Apache Log4j2 2.0-beta9 through 2.15.0 JNDI features allow remote code execution.'),
    ('CVE-2017-0144 (EternalBlue)',
     'Microsoft SMBv1 server in Windows 7, Windows Server 2008 R2 allows remote code execution.'),
    ('CVE-2014-0160 (Heartbleed)',
     'The TLS heartbeat extension in OpenSSL 1.0.1 before 1.0.1g does not properly validate '
     'packet length, allowing attackers to obtain sensitive information from process memory.'),
    ('CVE-2021-26855 (ProxyLogon)',
     'Microsoft Exchange Server 2013 through 2019 allows server-side request forgery.'),
]

rows = []
for name, text in FAMOUS_CVES:
    r = predictor.predict(text)
    rows.append({
        'CVE'       : name,
        'Vendor'    : ', '.join(r.entities.get('VENDOR',  [])),
        'Product'   : ', '.join(r.entities.get('PRODUCT', [])),
        'Version'   : ', '.join(r.entities.get('VERSION', [])),
        'CPE'       : r.cpe_string,
        'Confidence': f'{r.confidence:.2%}' if r.confidence else '—',
    })

pd.DataFrame(rows).set_index('CVE')

---
## 10. Batch Inference

Run inference on a list of CVE descriptions and export to CSV.

In [ ]:
import json

# Take the first N raw CVEs for demo batch inference
BATCH_N = 200

raw_file = f'{DATA_DIR}/raw/cves_{START_YEAR}_{END_YEAR}.jsonl'
sample_cves = []
with open(raw_file) as f:
    for i, line in enumerate(f):
        if i >= BATCH_N:
            break
        sample_cves.append(json.loads(line))

texts = [c.get('description', '') for c in sample_cves]
print(f'Running batch inference on {len(texts)} CVEs...')

from tqdm.notebook import tqdm
results_batch = []
for text in tqdm(texts, desc='Inferring'):
    r = predictor.predict(text)
    results_batch.append(r)

print(f'Done. {len(results_batch)} predictions.')

In [ ]:
rows = []
for cve, r in zip(sample_cves, results_batch):
    rows.append({
        'cve_id'    : cve.get('id', ''),
        'vendor'    : '|'.join(r.entities.get('VENDOR',  [])),
        'product'   : '|'.join(r.entities.get('PRODUCT', [])),
        'version'   : '|'.join(r.entities.get('VERSION', [])),
        'update'    : '|'.join(r.entities.get('UPDATE',  [])),
        'edition'   : '|'.join(r.entities.get('EDITION', [])),
        'cpe_string': r.cpe_string,
        'confidence': round(r.confidence, 4) if r.confidence else None,
        'description': r.text[:120] + '...' if len(r.text) > 120 else r.text,
    })

batch_df = pd.DataFrame(rows)

csv_path = f'{REPO_DIR}/predictions_{MODEL}.csv'
batch_df.to_csv(csv_path, index=False)
print(f'Saved {len(batch_df)} predictions to {csv_path}')

# Preview
batch_df[['cve_id','vendor','product','version','cpe_string','confidence']].head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confidence distribution
confs = batch_df['confidence'].dropna()
axes[0].hist(confs, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(confs.mean(), color='red', linestyle='--', label=f'Mean: {confs.mean():.3f}')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution')
axes[0].legend()

# Top vendors
all_vendors = [v for vendors in batch_df['vendor'].dropna() for v in vendors.split('|') if v]
from collections import Counter
top_vendors = Counter(all_vendors).most_common(10)
if top_vendors:
    names, cnts = zip(*top_vendors)
    axes[1].barh(names[::-1], cnts[::-1], color='#FF6B6B')
    axes[1].set_xlabel('Count')
    axes[1].set_title('Top 10 Vendors in Batch')

plt.tight_layout()
plt.show()

---
## 11. Multi-Model Comparison

Compare BERT, XLNet, and GPT-2 results side by side.
Only available if `TRAIN_ALL = True` was set in section 6.

In [ ]:
comparison_rows = []
for m in ['bert', 'xlnet', 'gpt2']:
    rpath = f'{MODELS_DIR}/{m}_ner/results.json'
    if not os.path.exists(rpath):
        continue
    with open(rpath) as f:
        r = json.load(f)
    tm = r['test_metrics']
    comparison_rows.append({
        'Model'    : m.upper(),
        'F1'       : tm.get('f1', 0),
        'Precision': tm.get('precision', 0),
        'Recall'   : tm.get('recall', 0),
        'Accuracy' : tm.get('accuracy', 0),
    })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows).set_index('Model')
    display(comp_df.style.format('{:.4f}').highlight_max(color='lightgreen'))

    # Radar / grouped bar
    metrics = ['F1', 'Precision', 'Recall', 'Accuracy']
    x = range(len(metrics))
    width = 0.25
    model_colors = {'BERT': 'steelblue', 'XLNET': 'seagreen', 'GPT2': 'tomato'}

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (model_name, row) in enumerate(comp_df.iterrows()):
        offset = (i - len(comp_df)/2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], [row[m] for m in metrics],
                      width=width, label=model_name,
                      color=model_colors.get(model_name, 'gray'))

    ax.set_xticks(list(x))
    ax.set_xticklabels(metrics)
    ax.set_ylim(0.85, 1.01)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Test Set Metrics')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No trained models found for comparison. Run section 6 with TRAIN_ALL=True.')

---
## 12. Run Evaluate Script (Detailed Demo)

In [ ]:
DEMO_TEXT = "Microsoft Windows 10 Version 21H2 is vulnerable to remote code execution via SMBv3."

!python scripts/evaluate.py \
    --model      {MODEL}          \
    --checkpoint {CHECKPOINT_DIR} \
    --text       "{DEMO_TEXT}"

---
## 13. Save Artifacts to Google Drive

In [ ]:
import shutil

if USE_DRIVE and DRIVE_ROOT:
    # Save trained model checkpoint
    dst_ckpt = os.path.join(DRIVE_ROOT, f'models/{MODEL}_ner')
    os.makedirs(dst_ckpt, exist_ok=True)
    if os.path.exists(f'{MODELS_DIR}/{MODEL}_ner'):
        shutil.copytree(f'{MODELS_DIR}/{MODEL}_ner', dst_ckpt, dirs_exist_ok=True)
        print(f'Model checkpoint saved to Drive: {dst_ckpt}')

    # Save batch predictions CSV
    if os.path.exists(csv_path):
        dst_csv = os.path.join(DRIVE_ROOT, f'predictions_{MODEL}.csv')
        shutil.copy(csv_path, dst_csv)
        print(f'Predictions saved to Drive: {dst_csv}')

    # Save TensorBoard logs
    if os.path.exists(f'{LOGS_DIR}/{MODEL}_ner'):
        dst_logs = os.path.join(DRIVE_ROOT, f'logs/{MODEL}_ner')
        shutil.copytree(f'{LOGS_DIR}/{MODEL}_ner', dst_logs, dirs_exist_ok=True)
        print(f'TensorBoard logs saved to Drive: {dst_logs}')
else:
    print('Drive not mounted — skipping Drive save.')
    print('Download files manually via Files panel (left sidebar → folder icon).')

In [ ]:
# Download predictions CSV directly to your computer
try:
    from google.colab import files
    if os.path.exists(csv_path):
        files.download(csv_path)
    results_json_path = f'{MODELS_DIR}/{MODEL}_ner/results.json'
    if os.path.exists(results_json_path):
        files.download(results_json_path)
except Exception as e:
    print(f'Manual download: {csv_path}')

---
## 14. Interactive Prediction Widget

Type any CVE description and press Enter to extract CPE entities.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Textarea(
    value='Apache Struts 2.3.5 through 2.3.31 and 2.5 through 2.5.10 is vulnerable to RCE.',
    layout=widgets.Layout(width='100%', height='80px'),
    description='CVE Text:',
    style={'description_width': '80px'},
)
run_btn = widgets.Button(description='Extract CPE', button_style='primary')
output  = widgets.Output()

def on_click(_):
    with output:
        clear_output(wait=True)
        r = predictor.predict(text_input.value)
        display(HTML(render_bio(r.tokens, r.bio_labels)))
        legend = ' '.join(
            f'<span style="background:{c};padding:2px 6px;border-radius:3px;font-size:11px">{e}</span>'
            for e, c in ENTITY_COLORS.items()
        )
        display(HTML(legend))
        print('\nEntities:')
        for k, vs in r.entities.items():
            print(f'  {k}: {vs}')
        print(f'\nCPE     : {r.cpe_string}')
        print(f'Confidence: {r.confidence:.2%}')

run_btn.on_click(on_click)
display(widgets.VBox([text_input, run_btn, output]))

---
## Summary

| Step | Script / Module | Output |
|------|----------------|--------|
| 1. Download | `scripts/download_data.py` | `data/raw/cves_*.jsonl` |
| 2. Annotate | `src/data/annotator.py` | `data/annotated/cves_*.bio` |
| 3. Train | `scripts/train.py` | `models/{model}_ner/best/` |
| 4. Evaluate | `scripts/evaluate.py` | `models/{model}_ner/results.json` |
| 5. Infer | `src/inference/predictor.py` | `PredictionResult` |

**Paper results (20 epochs, full dataset D5 ≈ 361k sentences):**

| Model | F1 | Accuracy |
|-------|----|----------|
| **BERT** | **95.48%** | **99.13%** |
| XLNet | 94.21% | 98.87% |
| GPT-2 | 91.43% | 98.34% |